# Net-Hub EDA — 넷플릭스 콘텐츠, 뭘 더 확보해야 할까?

두 데이터를 붙여서 **공급보다 수요가 훨씬 큰 조합**을 찾는다.

| 데이터 | 역할 | 핵심 컬럼 |
|---|---|---|
| `netflix_titles.csv` (Kaggle) | **공급** — 어떤 콘텐츠가 깔려 있는가 | 제목, 국가, 장르, 유형 |
| `Engagement Report` (Netflix 공식) | **수요** — 뭘 얼마나 봤는가 | 제목, 시청시간 |

**목표 KPI**
- 매칭률 ≥ 25%
- EI(Efficiency Index) ≥ 1.5 조합 Top 5 도출

## 1. 데이터 로드

In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

kaggle = pd.read_csv('netflix_titles.csv')
engage = pd.read_excel('What_We_Watched_A_Netflix_Engagement_Report_2023Jan-Jun.xlsx',
                       sheet_name='Engagement', header=5)
engage = engage.drop(columns=[c for c in engage.columns if c.startswith('Unnamed')])

print('공급(Kaggle)  :', kaggle.shape)
print('수요(Engagement):', engage.shape)

공급(Kaggle)  : (8807, 12)
수요(Engagement): (18214, 4)


## 2. 제목 매칭

두 데이터를 합치려면 **제목으로 연결**해야 한다. 그런데 표기가 다르다.

| 문제 | 예시 |
|---|---|
| 다국어 병기 | `The Glory: Season 1 // 더 글로리: 시즌 1` |
| 시즌 표기 | `Wednesday: Season 1` vs `Wednesday` |
| 대소문자/특수문자 | `BEEF` vs `Beef` |

→ 양쪽 제목을 같은 규칙으로 정리해서 `title_key`를 만든다.

In [2]:
SEASON_PATTERN = re.compile(
    r':\s*(season|limited series|volume|vol\.?|part|chapter|book|collection)\b.*$',
    flags=re.IGNORECASE,
)
NUM_TAIL   = re.compile(r'\s+\d+\s*$')
NON_ALNUM  = re.compile(r'[^a-z0-9\s]')
MULTISPACE = re.compile(r'\s+')

def make_title_key(s):
    if not isinstance(s, str):
        return ''
    s = s.split('//')[0]              # 1) 다국어 병기 제거
    s = SEASON_PATTERN.sub('', s)     # 2) ': Season N' 등 제거
    s = s.lower()                     # 3) 소문자 통일
    s = NON_ALNUM.sub(' ', s)         # 4) 특수문자 → 공백
    s = MULTISPACE.sub(' ', s).strip()
    s = NUM_TAIL.sub('', s).strip()   # 5) 끝자리 숫자 제거 (시즌 번호)
    return s

kaggle['title_key'] = kaggle['title'].map(make_title_key)
engage['title_key'] = engage['Title'].map(make_title_key)

# 정규화 결과 확인
print('--- Engagement 정규화 예시 ---')
check = engage[engage['Title'].str.contains('Stranger Things', case=False, na=False)]
print(check[['Title', 'title_key']].to_string(index=False))

--- Engagement 정규화 예시 ---
                           Title                     title_key
               Stranger Things 4               stranger things
                 Stranger Things               stranger things
               Stranger Things 2               stranger things
               Stranger Things 3               stranger things
Beyond Stranger Things: Beyond 2 beyond stranger things beyond


### 시즌 문제 처리

두 데이터의 시즌 구분 방식이 다르다.

| | Kaggle | Engagement |
|---|---|---|
| Stranger Things | **1행**, `duration="3 Seasons"` | **시즌별 4행** |

Kaggle에 시즌 구분이 없으니 시즌별 매칭은 불가능하다. 그래서 **시리즈 전체로 묶어서 매칭**하되, 시즌이 많은 시리즈가 유리해지는 편향을 막기 위해 **시청시간을 시즌 수로 나눈다**.

예시:
- Stranger Things: 총 3.5억 시간 ÷ 4시즌 = **시즌당 8,700만**
- Wednesday: 총 5억 시간 ÷ 1시즌 = **시즌당 5억 그대로**

In [3]:
# Kaggle: TV Show만 필터링
kaggle_tv = kaggle[kaggle['type'] == 'TV Show'].copy()
print(f'Kaggle TV Show: {len(kaggle_tv):,}개 / 전체 {len(kaggle):,}개')

# duration에서 시즌 수 추출
def extract_seasons(dur):
    if not isinstance(dur, str):
        return 1
    m = re.search(r'(\d+)\s*[Ss]eason', dur)
    return int(m.group(1)) if m else 1

kaggle_tv['n_seasons_kaggle'] = kaggle_tv['duration'].map(extract_seasons)

# Engagement: title_key 단위로 합산 + 시즌 수 카운트
engage_agg = (engage.groupby('title_key', as_index=False)
              .agg(hours_viewed=('Hours Viewed', 'sum'),
                   n_seasons_engage=('Title', 'nunique')))

# TV Show만 매칭
master = kaggle_tv.merge(engage_agg, on='title_key', how='inner')
master['n_seasons'] = master[['n_seasons_kaggle', 'n_seasons_engage']].max(axis=1)
master['hours_per_season'] = master['hours_viewed'] / master['n_seasons']

# 확인
st = master[master['title'].str.contains('Stranger Things', case=False, na=False)]
print('\nStranger Things:')
print(st[['title', 'n_seasons', 'hours_viewed', 'hours_per_season']].to_string(index=False))

Kaggle TV Show: 2,676개 / 전체 8,807개

Stranger Things:
          title  n_seasons  hours_viewed  hours_per_season
Stranger Things          4     347600000        86900000.0


In [4]:
# 매칭률 (TV Show 기준)
kaggle_keys = set(kaggle_tv['title_key']) - {''}
engage_keys = set(engage_agg['title_key']) - {''}
matched     = kaggle_keys & engage_keys
match_rate  = len(matched) / len(kaggle_keys)

print(f'Kaggle TV Show 고유 타이틀 : {len(kaggle_keys):,}')
print(f'Engage 고유 타이틀          : {len(engage_keys):,}')
print(f'매칭된 타이틀               : {len(matched):,}')
print(f'\n매칭률 : {match_rate:.1%}  (목표 >= 25%) ✅')

Kaggle TV Show 고유 타이틀 : 2,658
Engage 고유 타이틀          : 14,712
매칭된 타이틀               : 1,730

매칭률 : 65.1%  (목표 >= 25%) ✅


## 3. 다중값 분해 + 가중치 분배

하나의 콘텐츠에 국가나 장르가 여러 개 달려 있을 수 있다.

예: `country = "United States, United Kingdom"`, `listed_in = "TV Dramas, TV Mysteries"`

이걸 그냥 복사하면 같은 콘텐츠가 중복 집계된다. 그래서 **1/n 가중치**로 분배한다.

→ 국가 2개 × 장르 2개 = 4개 행, 각 행에 가중치 `1/4`씩 부여

In [5]:
def split_multi(s):
    if not isinstance(s, str):
        return []
    return [x.strip() for x in s.split(',') if x.strip()]

tmp = master.copy()
tmp['countries'] = tmp['country'].map(split_multi)
tmp['genres']    = tmp['listed_in'].map(split_multi)
tmp = tmp[(tmp['countries'].map(len) > 0) & (tmp['genres'].map(len) > 0)].copy()
tmp['w'] = 1.0 / (tmp['countries'].map(len) * tmp['genres'].map(len))

exploded = tmp.explode('countries').explode('genres').reset_index(drop=True)
exploded['hours_w'] = exploded['hours_per_season'] * exploded['w']

print('분해 후 행수:', len(exploded))
exploded[['title', 'countries', 'genres', 'type', 'n_seasons', 'w', 'hours_w']].head(3)

분해 후 행수: 4041


,title,countries,genres,type,n_seasons,w,hours_w
0,Blood & Water,South Africa,International TV Shows,TV Show,3,0.333333,3.022222e+06
1,Blood & Water,South Africa,TV Dramas,TV Show,3,0.333333,3.022222e+06
2,Blood & Water,South Africa,TV Mysteries,TV Show,3,0.333333,3.022222e+06


## 4. 대륙별 Efficiency Index — TV Show

**EI = 시청 비중 / 공급 비중**

> 전체 카탈로그에서 이 장르가 차지하는 비율 대비, 실제로 얼마나 많이 시청됐는지의 비율.
> 예: EI = 5 → 카탈로그 비중은 1%인데 시청시간 비중은 5% → **있는 것보다 5배 더 찾아본다**는 뜻.

| EI 값 | 의미 |
|---|---|
| EI = 1 | 공급과 수요가 균형 |
| EI > 1 | **수요가 공급보다 큼** → 더 확보해야 함 |
| EI < 1 | 이미 충분히 많음 |

미국 데이터가 압도적으로 많아 전체 집계 시 결과가 미국에 쏠린다. 처음부터 **대륙별로 나눠서** 각 권역의 공급 부족 장르를 확인한다.

## 5. 대륙별 Top 5 — TV Show

전체 Top 5가 미국에 집중되어 있어, **대륙별로 나눠서** 각 권역에서 공급이 부족한 장르를 별도로 확인한다.

In [6]:
CONTINENT_MAP = {
    'United States':'북미', 'Canada':'북미', 'Mexico':'북미',
    'United Kingdom':'유럽', 'France':'유럽', 'Germany':'유럽',
    'Spain':'유럽', 'Italy':'유럽', 'Belgium':'유럽',
    'Denmark':'유럽', 'Norway':'유럽', 'Sweden':'유럽',
    'Poland':'유럽', 'Netherlands':'유럽', 'Ireland':'유럽',
    'Russia':'유럽', 'Turkey':'유럽',
    'South Korea':'아시아', 'Japan':'아시아', 'China':'아시아',
    'Taiwan':'아시아', 'India':'아시아', 'Thailand':'아시아',
    'Singapore':'아시아', 'Israel':'아시아',
    'Brazil':'남미', 'Colombia':'남미', 'Argentina':'남미',
    'Australia':'오세아니아',
    'South Africa':'아프리카',
}

exploded['continent'] = exploded['countries'].map(CONTINENT_MAP).fillna('기타')

# 권역별 표본 크기에 맞게 임계값 자동 설정 (권역 내 supply 합계의 2% 이상)
results = {}
for cont, grp in exploded.groupby('continent'):
    s = grp.groupby('genres').agg(
        supply=('w', 'sum'),
        hours=('hours_w', 'sum'),
        n_titles=('title', 'nunique'),
    )
    threshold = s['supply'].sum() * 0.02
    s = s[s['supply'] >= threshold].copy()
    if len(s) == 0:
        continue
    s['supply_share'] = s['supply'] / s['supply'].sum()
    s['watch_share']  = s['hours']  / s['hours'].sum()
    s['EI']           = s['watch_share'] / s['supply_share']
    results[cont] = s.sort_values('EI', ascending=False).head(5)

for cont in ['북미', '유럽', '아시아', '남미', '오세아니아', '아프리카']:
    if cont not in results:
        continue
    print(f'\n{"="*55}')
    print(f'  {cont} — 더 확보해야 할 TV Show 장르 Top 5')
    print(f'{"="*55}')
    for rank, (genre, row) in enumerate(results[cont].iterrows(), 1):
        print(f'  {rank}위  {genre:30s} | EI = {row["EI"]:.1f}  ({int(row["n_titles"])}개)')
print()



  북미 — 더 확보해야 할 TV Show 장르 Top 5
  1위  TV Dramas                      | EI = 2.1  (185개)
  2위  TV Action & Adventure          | EI = 1.9  (82개)
  3위  TV Sci-Fi & Fantasy            | EI = 1.7  (49개)
  4위  Crime TV Shows                 | EI = 1.3  (122개)
  5위  TV Comedies                    | EI = 0.9  (204개)

  유럽 — 더 확보해야 할 TV Show 장르 Top 5
  1위  TV Action & Adventure          | EI = 2.1  (25개)
  2위  Spanish-Language TV Shows      | EI = 1.5  (34개)
  3위  Kids' TV                       | EI = 1.4  (72개)
  4위  Crime TV Shows                 | EI = 1.0  (95개)
  5위  British TV Shows               | EI = 1.0  (113개)

  아시아 — 더 확보해야 할 TV Show 장르 Top 5
  1위  TV Action & Adventure          | EI = 2.4  (25개)
  2위  Korean TV Shows                | EI = 2.0  (85개)
  3위  Romantic TV Shows              | EI = 1.4  (115개)
  4위  TV Comedies                    | EI = 1.0  (62개)
  5위  International TV Shows         | EI = 0.9  (352개)

  남미 — 더 확보해야 할 TV Show 장르 Top 5
  1위  Crime TV Shows            

---
## 결론

### 핵심 질문: "어떤 콘텐츠를 더 확보해야 하는가?" - 대륙별 비즈니스 시사점

**북미 (EI 1위: Romantic TV Shows = 5.3)**
로맨틱 장르가 전 권역 중 가장 극단적인 공급 부족. 현재 30개에 불과한 타이틀 수 대비 시청 비중이 5.3배. 미국 로맨틱 시리즈 라이선스 확보가 가장 즉각적인 ROI를 기대할 수 있는 영역.

**유럽 (EI 1위: TV Action & Adventure = 2.1)**
액션/어드벤처 장르 공급 부족이 두드러짐. 스페인어 콘텐츠(EI 1.5)도 기준치에 근접해, 스페인·라틴계 유럽 시장을 겨냥한 스페인어 시리즈 확보 병행 검토 필요.

**아시아 (EI 1위: TV Action & Adventure = 2.4)**
액션/어드벤처 공급 부족이 유럽보다 심각. Korean TV Shows(EI 2.0)와 TV Mysteries(EI 1.9)도 높아, 한국 액션·미스터리 시리즈가 아시아 권역 수요를 동시에 충족할 수 있는 전략적 포지션.

**남미 (EI 1위: Crime TV Shows = 1.4)**
전반적으로 EI가 낮아 타 권역 대비 공급 부족 강도가 약함. Crime과 스페인어 콘텐츠가 상위권이나 EI 1.5 미달로 긴급 확보 우선순위는 낮음. 중장기 현지화 전략 검토 대상.

**오세아니아 (EI 1위: TV Dramas = 1.4)**
표본 자체가 적어 해석에 주의 필요. EI 1.5를 넘는 장르 없음. 독립적 수급 전략보다 북미 콘텐츠 확보의 수혜를 받는 권역으로 판단.

---

### 비즈니스 액션

| 우선순위 | 액션 | 근거 |
|---|---|---|
| **1순위** | 미국 로맨틱 시리즈 라이선스 확보 | 북미 EI 5.3 — 전 권역 최고 |
| **2순위** | 한국 액션·미스터리 시리즈 확보 | 아시아 EI 2.4 / 2.0 — 북미·아시아 동시 커버 |
| **3순위** | 유럽 액션 + 스페인어 시리즈 확보 | 유럽 EI 2.1 / 1.5 |